In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from linear_regression_ny import LinearRegression

data = np.genfromtxt("C:/Users/samue/Documents/code/Statistiska-metoder-Samuel-Airisniemi/Labb_ny/housing.csv", delimiter = ",", dtype = str, skip_header = 1)
data = np.char.strip(data)
data = data[~np.any(data == "", axis = 1)]

print("Dataset shape:", data.shape)
print("First row:", data[0])

### Grundläggande dataexploration (EDA)

Datasetet innehåller ursprungligen 20 640 observationer och 10 variabler.  
En variabel är kategorisk (ocean_proximity) och övriga är numeriska.

Variabeln total_bedrooms innehåller 207 saknade värden.  
I denna analys tas dessa rader bort före modellering, vilket ger ett slutligt stickprov på 20 433 observationer.

Detta innebär att OLS-modellerna skattas på ett komplett datamaterial utan saknade värden, men att stickprovsstorleken blir något mindre än i originaldatasetet.

In [ ]:
df = pd.read_csv("C:/Users/samue/Documents/code/Statistiska-metoder-Samuel-Airisniemi/Labb_ny/housing.csv")
df.info()

In [ ]:
df.describe()

In [ ]:
df['ocean_proximity'].value_counts()

### Outliers och capped values

Variabeln median_house_value är capped vid 500001 USD.  
Det innebär att alla värden över denna nivå har klippts av och inte representerar verkliga priser.

I denna analys väljer jag att behålla dessa observationer eftersom:

- De utgör en relativt liten del av datasetet  
- De fortfarande innehåller relevant information om områden med höga priser  
- Jag vill undvika att minska stickprovsstorleken i onödan  

Det viktiga är att vara medveten om denna begränsning när resultaten tolkas, eftersom cap-värdet kan innebära att de högsta bostadspriserna underskattas i modellen.

### Modellstrategi

Jag börjar med en ren numerisk modell (Model 1) för att:

1. Få en baslinje för modellens förklaringsgrad  
2. Identifiera kollinearitet mellan numeriska variabler  
3. Se vilka variabler som är signifikanta utan kategorisk information  

Därefter utökar jag modellen med kategorivariabeln OceanProximity (Model 2).  
Slutligen bygger jag en feature‑engineered modell (Model 3) för att minska kollinearitet och öka tolkbarheten.

In [ ]:
X_num = data[:, 0:8].astype(float)
y = data[:, 8].astype(float)

feature_names_num = ["Longitude", "Latitude", "HouseAge", "TotalRooms", "TotalBedrooms", "Population", "Households", "MedInc"]

model1 = LinearRegression()
model1.fit(X_num, y, feature_names = feature_names_num)

print("MODEL 1 (Numeric only):")
model1.summary(confidence_level = 0.95)
print("R^2:", model1.r2())
print("Adjusted R^2:", model1.adjusted_r2())
print("RMSE:", model1.rmse())

print("\nVariance Inflation Factors (Model 1)")
vif_values = model1.vif(X_num)
for name, vif in zip(feature_names_num, vif_values):
    print(f"{name:25s} {vif:.2f}")

### Tolkning av Model 1

Model 1 fungerar som en basmodell med enbart numeriska variabler. Den ger en relativt hög förklaringsgrad, men VIF-värdena visar att flera av de storleksrelaterade variablerna lider av stark multikollinearitet.

F-testet visar att modellen som helhet är statistiskt signifikant, vilket innebär att minst en av förklaringsvariablerna bidrar till att förklara variationen i responsvariabeln.

In [ ]:
np.set_printoptions(precision = 3, suppress = True)
pearson = model1.pearson_matrix(X_num)

print("Pearson correlation matrix:")
print(pearson)

plt.imshow(pearson, cmap = "coolwarm", vmin = -1, vmax = 1)
plt.colorbar()
plt.title("Pearson Correlation Matrix (Numeric Features)")
plt.xticks(range(len(feature_names_num)), feature_names_num, rotation = 95)
plt.yticks(range(len(feature_names_num)), feature_names_num)
plt.tight_layout()
plt.show()

### Kollinearitet

Pearson-korrelationen mellan variablerna visar att flera av de numeriska variablerna är starkt korrelerade, särskilt: TotalRooms, TotalBedrooms, Population, Households.

Dessa variabler beskriver i stor utsträckning samma underliggande egenskaper hos bostadsområdena. Denna typ av multikollinearitet påverkar inte modellens prediktiva förmåga lika mycket, men den kan göra koefficienterna mindre stabila och svårare att tolka.

Detta bekräftas också genom VIF-analysen. Särskilt TotalBedrooms och Households har mycket höga VIF-värden, vilket indikerar stark multikollinearitet. Detta innebär att de individuella t-testerna bör tolkas försiktigt, eftersom flera variabler delar samma förklaringsinformation.

När kollinearitet är stark kan signifikanstesterna bli mindre stabila, eftersom variablerna delar information och därmed konkurrerar om samma förklaringskraft i modellen. Detta kan leda till större standardfel och göra koefficienternas signifikans mer känslig för små förändringar i datamaterialet.

Detta innebär att även om modellen kan ha god prediktiv förmåga kan de individuella koefficienterna vara instabila och känsliga för små förändringar i datamaterialet.

### Är alla parametrar lämpliga att ha med?

Alla parametrar som är statistiskt signifikanta är inte automatiskt lämpliga att ha med i modellen.

Detta innebär att vissa av de absoluta storleksvariablerna inte är optimala att inkludera samtidigt i en regressionsmodell om målet är statistisk tolkning. När variabler beskriver nästan samma information kan deras individuella koefficienter bli instabila och svåra att tolka isolerat.

I den första modellen inkluderas flera variabler som är starkt korrelerade med varandra. Detta gör att modellen innehåller viss redundans. Även om koefficienterna kan vara signifikanta blir deras tolkning svårare eftersom förändringar i en variabel ofta sammanfaller med förändringar i en annan.

I den tredje modellen ersätts därför flera absoluta variabler med kvotvariabler såsom: RoomsPerHousehold, BedroomsPerRoom, PopulationPerHousehold.

Dessa variabler fångar liknande information men minskar redundansen mellan variablerna och ger därför en modell som är lättare att tolka.

Det gäller även vissa kategoriska parametrar. Exempelvis har kategorin ISLAND mycket få observationer i datamaterialet, vilket gör att dess koefficient bör tolkas försiktigt även om den blir statistiskt signifikant.

I Model 1 ligger VIF-värdena särskilt högt för TotalBedrooms och Households, vilket tyder på stark multikollinearitet och gör de individuella koefficienterna svårare att tolka isolerat.

In [ ]:
feature_names_full = feature_names_num + ["OceanProximity"]

X_raw = np.column_stack((data[:, :8], data[:, 9]))
categorical_cols = [8]

model2 = LinearRegression()
X_encoded, new_names = model2.one_hot_encode(X_raw, categorical_cols = categorical_cols, feature_names = feature_names_full, drop_first = True)

model2.fit(X_encoded, y, feature_names = new_names)

print("MODEL 2 (Numeric + One-Hot Encoded Categorical):")
model2.summary(confidence_level = 0.95)

print("R^2:", model2.r2())
print("Adjusted R^2:", model2.adjusted_r2())
print("RMSE:", model2.rmse())

print("\nVariance Inflation Factors (Model 2)")
vif_values = model2.vif(X_encoded)
for name, vif in zip(new_names, vif_values):
    print(f"{name:25s} {vif:.2f}")

### Tolkning av kategoriska variabler i Model 2

I Model 2 används one-hot-kodning av ocean_proximity med en referenskategori som utelämnas för att undvika perfekt multikollinearitet. Koefficienterna för de kvarvarande kategorierna ska därför tolkas relativt referenskategorin.

Referenskategorin i modellen är <1H OCEAN, vilket innebär att koefficienterna för de övriga kategorierna visar skillnaden i pris relativt denna kategori.

In [ ]:
rooms_per_household = X_num[:, 3] / X_num[:, 6]
bedrooms_per_room = X_num[:, 4] / X_num[:, 3]
population_per_household = X_num[:, 5] / X_num[:, 6]

lon = X_num[:, 0]
lat = X_num[:, 1]
lon0 = np.mean(lon)
lat0 = np.mean(lat)

distance_to_centroid = np.sqrt((lon - lon0)**2 + (lat - lat0)**2)

X_fe = np.column_stack([lon, lat, X_num[:, 2], X_num[:, 7], rooms_per_household, bedrooms_per_room, population_per_household, distance_to_centroid])

feature_names_fe = ["Longitude", "Latitude", "HouseAge", "MedInc", "RoomsPerHousehold", "BedroomsPerRoom", "PopulationPerHousehold", "DistanceToCentroid"]

model3 = LinearRegression()
model3.fit(X_fe, y, feature_names = feature_names_fe)

print("MODEL 3 (Feature engineered, reduced collinearity):")
model3.summary(confidence_level = 0.95)

print("R^2:", model3.r2())
print("Adjusted R^2:", model3.adjusted_r2())
print("RMSE:", model3.rmse())

print("\nVariance Inflation Factors (Model 3)")
vif_values = model3.vif(X_fe)
for name, vif in zip(feature_names_fe, vif_values):
    print(f"{name:25s} {vif:.2f}")

### Tolkning av Model 3

Model 3 använder feature engineering där flera av de ursprungliga storleksvariablerna ersätts med kvotvariabler som RoomsPerHousehold, BedroomsPerRoom och PopulationPerHousehold. Syftet är att minska redundans mellan variabler som tidigare visade stark multikollinearitet.

Resultaten visar att modellen får en något lägre förklaringsgrad än Model 2, men variablerna blir mer tolkbara eftersom de beskriver relationer mellan bostadsegenskaper snarare än absoluta storlekar.

Model 3 kan därför ses som en mer tolkbar modell med lägre redundans mellan variabler, även om den inte har lika hög förklaringsgrad som Model 2.

Dessa kvotvariabler minskar multikollineariteten eftersom de sammanfattar relationer mellan variabler som tidigare var starkt korrelerade.

### Vad säger förklaringsgraden om modellen?

Model 1: R² = 0.6369  
Model 2: R² = 0.6465  
Model 3: R² = 0.6158  

Ett högre R² innebär dock inte nödvändigtvis att modellen är bäst för statistisk tolkning. Om modellen innehåller stark multikollinearitet kan koefficienterna fortfarande vara svåra att tolka även om förklaringsgraden är hög.

R² visar hur stor andel av variationen i responsvariabeln median_house_value som förklaras av modellens förklaringsvariabler.

Model 2 har högst förklaringsgrad och förklarar cirka 64.6 % av variationen i responsvariabeln. Samtidigt innebär detta att ungefär 35 % av variationen fortfarande lämnas oförklarad, vilket visar att modellen är användbar men inte fullständig.

Model 3 har lägre förklaringsgrad än både Model 1 och Model 2. Det betyder att den är svagare i ren modellanpassning, även om den är mer lättolkad.

Förklaringsgraden talar därför för att Model 2 är den starkaste modellen totalt sett, medan Model 3 främst är intressant som en mer tolkbar alternativ modell.

In [ ]:
ci_95 = model2.confidence_intervals(confidence_level = 0.95)

print("Exempel på 95% konfidensintervall för de första parametrarna:")
for i, name in enumerate(model2.feature_names[:5]):
    print(f"{name:25s} 95%: [{ci_95[i][0]:.3f}, {ci_95[i][1]:.3f}]")

### Konfidensintervall

I denna analys rapporteras 95 % konfidensintervall, vilket är ett etablerat standardval i statistisk inferens. I denna uppgift finns det inte tillräckligt stöd för att motivera en lägre konfidensnivå som 90 %, och därför används 95 % som utgångspunkt.

Konfidensintervallen bör dock inte tolkas isolerat. Eftersom modellen har måttlig förklaringsgrad och vissa variabler uppvisar multikollinearitet, bör intervallen bedömas tillsammans med p-värden, R² och analysen av kollinearitet.

### Modelljämförelse

De tre modellerna visar olika egenskaper:

Model 1 innehåller alla numeriska variabler och lider av tydlig multikollinearitet. Model 2 har den högsta förklaringsgraden och beskriver därför datamaterialet bäst i termer av total variation. Samtidigt visar VIF-analysen att modellen fortfarande innehåller betydande multikollinearitet. Model 3 har något lägre förklaringsgrad men lägre redundans mellan variablerna, vilket gör den mer lämplig för tolkning av individuella effekter.

Sammanfattningsvis ger Model 2 den bästa förklaringsgraden, medan Model 3 har lägre redundans mellan variablerna och därför är mer lämplig för statistisk tolkning av individuella effekter.

### Slutsats

I denna labb har jag byggt flera linjära regressionsmodeller för att förklara variationen i huspriser i Kalifornien.

- Model 1 använde enbart numeriska variabler. Den gav en relativt hög förklaringsgrad, men led av tydlig multikollinearitet mellan flera storleksrelaterade variabler.  
- Model 2 utökade modellen med en kategorisk variabel ocean_proximity via one‑hot‑kodning. Detta ökade R² och visade att läget i förhållande till havet har en tydlig effekt på priset.  
- Model 3 använde feature engineering för att minska kollinearitet och skapa mer tolkbara variabler. Modellen blev lättare att tolka, men hade något lägre förklaringsgrad än Model 2.

Jag har analyserat:

- kollinearitet med hjälp av Pearson‑korrelation och diskuterat hur den påverkar t‑tester och tolkning av koefficienter  
- signifikans med både F‑test (hela modellen) och t‑test (enskilda parametrar)  
- konfidensintervall med 95 % konfidensnivå och diskuterat hur dessa bör tolkas tillsammans med p-värden, R² och multikollinearitet  
- förklaringsgrad (R²) och hur den ska tolkas i relation till modellens stabilitet

Sammantaget visar analysen att den modell som har högst förklaringsgrad inte nödvändigtvis är den mest statistiskt informativa. Valet av modell bör därför baseras både på modellanpassning och på hur stabila och tolkbara koefficienterna är. Detta illustrerar den klassiska avvägningen mellan modellanpassning och inferentiell stabilitet i regressionsanalys.

Trots att Model 2 har den högsta förklaringsgraden innebär den kvarvarande multikollineariteten att vissa individuella koefficienter bör tolkas försiktigt. När flera variabler beskriver nästan samma underliggande egenskaper kan deras t-tester bli mindre stabila eftersom variablerna delar förklaringsinformation. Model 3 har något lägre R² men lägre redundans mellan variablerna, vilket gör koefficienterna mer tolkbara ur ett inferensperspektiv. Valet av modell bör därför inte enbart baseras på förklaringsgrad utan också på modellens stabilitet och tolkbarhet.

### Reflektion

Denna labb visar tydligt skillnaden mellan:

- en modell som bara fungerar numeriskt  
- och en modell som är statistiskt välgrundad  

Genom att analysera kollinearitet, signifikans, konfidensintervall och kategoriska variabler blir det tydligt att modellering inte bara handlar om att få ett högt R², utan om att förstå *varför* modellen beter sig som den gör.

Feature engineering och kritisk analys är centrala delar av statistisk modellering, och denna labb illustrerar hur dessa steg kan förbättra modellens tolkbarhet och minska redundans mellan variabler.